# Cats & Dogs bootstrap (CIFAR-10)

CIFAR-10 is tiny and still available. We grab cat/dog samples from it,
save a minimal folder structure under `data/cats_dogs/`, show a couple of
images, print label counts, and preload tiny text/image embedding models
for the attention experiments. Run from the repo root (ensure `data/` is
writable).

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image
import torchvision
from torchvision import transforms

DATA_ROOT = Path("data/cats_dogs")
CIFAR_ROOT = Path("data/cifar10")

DATA_ROOT.mkdir(parents=True, exist_ok=True)

transform = transforms.ToTensor()
train_set = torchvision.datasets.CIFAR10(
    root=str(CIFAR_ROOT), train=True, download=True, transform=transform
)
test_set = torchvision.datasets.CIFAR10(
    root=str(CIFAR_ROOT), train=False, download=True, transform=transform
)

cat_label = 3  # 'cat'
dog_label = 5  # 'dog'
label_names = {cat_label: "cat", dog_label: "dog"}


print("done")

In [ ]:
def counts_by_label(ds) -> Counter:
    return Counter(label for _, label in ds)

train_counts = counts_by_label(train_set)
test_counts = counts_by_label(test_set)
print("Train cats/dogs:", {k: train_counts[k] for k in label_names})
print("Test cats/dogs:", {k: test_counts[k] for k in label_names})

def first_k(ds, label, k):
    found = []
    for img, lbl in ds:
        if lbl == label:
            found.append(img)
            if len(found) >= k:
                break
    return found

cats = first_k(train_set, cat_label, 2)
dogs = first_k(train_set, dog_label, 2)

cats_dir = DATA_ROOT / "cats"
dogs_dir = DATA_ROOT / "dogs"
cats_dir.mkdir(parents=True, exist_ok=True)
dogs_dir.mkdir(parents=True, exist_ok=True)

cat_paths: list[Path] = []
dog_paths: list[Path] = []
for i, img in enumerate(cats):
    path = cats_dir / f"cat_{i}.png"
    transforms.ToPILImage()(img).save(path)
    cat_paths.append(path)
for i, img in enumerate(dogs):
    path = dogs_dir / f"dog_{i}.png"
    transforms.ToPILImage()(img).save(path)
    dog_paths.append(path)

print("Saved cat images:", cat_paths)
print("Saved dog images:", dog_paths)

sample_paths = cat_paths + dog_paths


In [ ]:
fig, axes = plt.subplots(1, len(sample_paths), figsize=(10, 3))
for ax, path in zip(axes, sample_paths):
    with Image.open(path) as img:
        ax.imshow(img)
    ax.set_title(path.parent.name)
    ax.axis("off")
plt.tight_layout()


In [ ]:
# Tiny embedding backbones: MobileNetV3-Small (image) + bert-tiny (text)
import torch
from torchvision import models
from transformers import AutoModel, AutoTokenizer

TEXT_MODEL_NAME = "prajjwal1/bert-tiny"
image_weights = models.MobileNet_V3_Small_Weights.DEFAULT
image_model = models.mobilenet_v3_small(weights=image_weights).eval()
image_preprocess = image_weights.transforms()

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).eval()

sample_texts = ["a photo of a cat", "a photo of a dog"]
with torch.no_grad():
    text_batch = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True)
    text_embeddings = text_model(**text_batch).last_hidden_state[:, 0]
print("Text CLS embeddings shape:", tuple(text_embeddings.shape))

sample_image_path = cat_paths[0]
with Image.open(sample_image_path).convert("RGB") as img:
    img_tensor = image_preprocess(img).unsqueeze(0)
    with torch.no_grad():
        image_features = image_model.features(img_tensor).mean(dim=[2, 3])
print("Image feature shape:", tuple(image_features.shape))
